# Operational Definition of Arthouse Cinema
## From Analytics to Action - DTU Spring 2026

**Case company:** Publikum (formerly Will & Agency)  
**Dataset:** `03-data/films_enriched.csv` (50,000 films, IMDb + TMDb + MovieLens enrichment)  
**Purpose:** Define what counts as "arthouse" in this project, compare operational candidates, and provide a reusable flag for downstream analysis.

---

### How to read this notebook

This notebook treats "arthouse" as an analytical working definition, not a timeless film-theory category. The goal is practical: give our project one reproducible way to segment the 50k film dataset so later analyses can focus on the films most relevant to Publikum's arthouse/festival positioning question.


## Table of Contents

1. [Setup and Imports](#1-setup)
2. [Dataset Audit](#2-dataset-audit)
3. [Literature and Project Framing](#3-framing)
4. [Candidate Signals in the Data](#4-signals)
5. [Candidate Definitions](#5-definitions)
6. [Comparative Analysis](#6-comparison)
7. [Sanity Checks: Examples, False Positives, False Negatives](#7-sanity-checks)
8. [Recommendation and Reusable Function](#8-recommendation)
9. [Open Questions and Known Limitations](#9-limitations)


<a id="1-setup"></a>
## 1. Setup and Imports

We start by making the notebook robust to being run from either the repository root or the notebook folder. The `MPLCONFIGDIR` line avoids Matplotlib trying to write cache files outside the sandboxed project environment during execution.


In [ ]:
# Standard libraries
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Make Matplotlib execution reliable in restricted environments.
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

# Find project root by walking upward until 03-data exists.
PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / '03-data').exists() and (candidate / 'src').exists():
        PROJECT_ROOT = candidate
        break
os.chdir(PROJECT_ROOT)

os.makedirs('reports/figures', exist_ok=True)

from src.visualize import set_style, save_figure
from src.arthouse import (
    ARTHOUSE_KEYWORDS,
    FESTIVAL_TERMS,
    MAINSTREAM_RISK_KEYWORDS,
    SPECIALTY_LABELS,
    arthouse_signals,
    contains_any_term,
    is_arthouse,
    score_arthouse,
)

set_style()
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 140)

print(f'Working directory: {Path.cwd()}')
print('Libraries and project helpers loaded successfully.')


In [ ]:
# Explicit auditable term lists used by the notebook. The canonical implementation
# lives in src/arthouse.py; these local bindings make the choices visible here.
NOTEBOOK_SPECIALTY_LABELS = ('a24', 'ifc films', 'ifc productions', 'janus films', 'criterion collection', 'criterion', 'neon', 'mubi', 'strand releasing', 'kino lorber', 'oscilloscope', 'sony pictures classics', 'film movement', 'music box films', 'magnolia pictures', 'zeitgeist films', 'artificial eye', 'curzon', 'mk2', 'celluloid dreams', 'films boutique', 'the match factory', 'wild bunch', 'good machine', 'new yorker films', 'milestone films', 'bfi', 'british film institute', 'cinema guild', 'grasshopper film', 'factory 25', 'kimstim', 'utopia', 'metrograph', 'sideshow', 'picturehouse', 'roadside attractions', 'bleecker street', 'cohen media', 'arrow films', 'icarus films', 'dogwoof', 'modern films', 'trigon-film')
NOTEBOOK_FESTIVAL_TERMS = ('cannes', 'sundance', 'venice', 'berlinale', 'berlin international film festival', 'toronto international film festival', 'tiff', 'locarno', 'rotterdam', 'tribeca', 'telluride', 'san sebastian', 'london film festival', 'film festival', "palme d'or", 'palme d’or', 'golden bear', 'golden lion', 'jury prize', 'festival')
NOTEBOOK_ARTHOUSE_KEYWORDS = ('art film', 'art documentary', 'avant-garde', 'avant garde', 'experimental film', 'experimental cinema', 'slow cinema', 'minimalism', 'surrealism', 'existentialism', 'alienation', 'loneliness', 'identity', 'human rights', 'refugee', 'racism', 'holocaust', 'genocide', 'war crime', 'dictatorship', 'political repression', 'blasphemy', 'homosexuality', 'lgbt', 'lgbtq', 'addiction', 'suicide', 'social realism', 'coming of age', 'world war ii', 'communism', 'poverty', 'immigration')
NOTEBOOK_MAINSTREAM_RISK_KEYWORDS = ('superhero', 'marvel', 'dc comics', 'sequel', 'franchise', 'blockbuster', 'based on comic book', 'explosion', 'car chase', 'alien invasion', 'disaster movie', 'zombie', 'slasher', 'gore', 'torture porn', 'exploitation', 'b-movie', 'erotica')

assert NOTEBOOK_SPECIALTY_LABELS == SPECIALTY_LABELS
assert NOTEBOOK_FESTIVAL_TERMS == FESTIVAL_TERMS
assert NOTEBOOK_ARTHOUSE_KEYWORDS == ARTHOUSE_KEYWORDS
assert NOTEBOOK_MAINSTREAM_RISK_KEYWORDS == MAINSTREAM_RISK_KEYWORDS

print('Notebook term lists match src.arthouse constants.')


<a id="2-dataset-audit"></a>
## 2. Dataset Audit

Constraint check: before defining anything, verify the columns we actually have. The analysis below uses only columns present in `films_enriched.csv`.


In [ ]:
DATA_PATH = Path('03-data/films_enriched.csv')
df = pd.read_csv(DATA_PATH)

required_columns = [
    'originalTitle', 'releaseYear', 'runtimeMinutes', 'imdbRating', 'numberOfVotes',
    'genres', 'keywords', 'production', 'budget', 'revenue', 'tmdb_popularity',
    'tmdb_vote_average', 'tmdb_vote_count', 'original_language', 'production_countries',
    'firstLanguage', 'mainCountry', 'directors', 'writers'
]
missing_columns = [col for col in required_columns if col not in df.columns]
assert not missing_columns, f'Missing required columns: {missing_columns}'

print(f'Dataset shape: {df.shape[0]:,} films x {df.shape[1]:,} columns')
print(f'Loaded from: {DATA_PATH}')

column_audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'non_null': df.notna().sum(),
    'null_pct': (df.isna().mean() * 100).round(2),
})
display(column_audit.loc[required_columns])


In [ ]:
# Key distribution checks used by the definitions.
rated = df[df['imdbRating'].notna() & df['numberOfVotes'].notna()]
rated_keyworded = rated[rated['keywords'].notna()]

quantile_table = pd.DataFrame({
    'imdbRating': rated['imdbRating'].quantile([0.10, 0.25, 0.50, 0.75, 0.90]),
    'numberOfVotes': rated['numberOfVotes'].quantile([0.10, 0.25, 0.50, 0.75, 0.90]),
    'tmdb_popularity': df['tmdb_popularity'].dropna().quantile([0.10, 0.25, 0.50, 0.75, 0.90]),
}).round(2)

pdf_vote_q25 = rated_keyworded['numberOfVotes'].quantile(0.25)
pdf_baseline_count = int(((rated_keyworded['imdbRating'] >= 7.0) & (rated_keyworded['numberOfVotes'] <= pdf_vote_q25)).sum())

print(f'Rated films: {len(rated):,}')
print(f'Rated + keyworded films: {len(rated_keyworded):,}')
print(f'Bottom-quartile IMDb votes among rated + keyworded films: {pdf_vote_q25:.0f}')
print(f'PDF baseline reproduction, rating >= 7.0 and votes <= {pdf_vote_q25:.0f}: {pdf_baseline_count:,} films')
display(quantile_table)


<a id="3-framing"></a>
## 3. Literature and Project Framing

### Project provocation recap

The motivating PDF, `notebooks/arthouse/arthouse_provocation_final.pdf`, asks: **Which provocative themes are good provocation for arthouse cinema?** Its existing operational move was:

- **Niche reach:** bottom quartile of IMDb vote count among rated + keyworded films, which this notebook verifies as `<= 33` votes.
- **Quality threshold:** `imdbRating >= 7.0`.
- **Result:** 1,673 arthouse films in the rated + keyworded subset.

That framing is useful because it treats arthouse as a **high-rating / low-reach sweet spot**: critically recognized films that have not reached a mass IMDb audience.

### External framing used here

- David Bordwell's "art cinema" framing treats art cinema as a mode with conventions such as realism, authorial expressivity, ambiguity, and distance from classical cause-effect Hollywood narration. Source: [EBSCO abstract for Bordwell, *The Art Cinema as a Mode of Film Practice*](https://openurl.ebsco.com/contentitem/gcd%3A31286128?id=ebsco%3Agcd%3A31286128&sid=ebsco%3Aplink%3Acrawler).
- Film festivals are a distribution and evaluation circuit for new or outstanding films, connecting filmmakers, distributors, critics, and audiences. Source: [Britannica, film festival](https://www.britannica.com/art/film-festival).

### Candidate operational traditions

1. **Specialty distribution:** arthouse as films handled by specialty labels and arthouse distributors.
2. **Festival circuit:** arthouse as films with festival launch, awards, or festival-market evidence.
3. **Auteur/formal mode:** arthouse as director-driven, ambiguous, experimental, slow, or formally unusual work.
4. **Low-budget non-commercial cinema:** arthouse as lower-budget film outside blockbuster economics.
5. **Foreign-language / non-Hollywood cinema:** arthouse as non-English and non-US cinema reaching niche international audiences.

The dataset cannot observe form directly, so each tradition below is mapped to imperfect but auditable proxies.


<a id="4-signals"></a>
## 4. Candidate Signals Available in the Data

The enriched dataset gives us distribution labels, keyword tags, language/country metadata, budget and popularity proxies, and IMDb rating/reach. The table maps theory or industry concepts to available columns.


In [ ]:
signal_map = pd.DataFrame([
    {
        'definition_family': 'Specialty distribution',
        'columns': 'production',
        'signal': 'Production/distribution string contains an explicit specialty label such as A24, IFC Films, Janus, Criterion, Neon, Mubi, Strand, Kino Lorber, Oscilloscope, Sony Pictures Classics, etc.',
        'caveat': 'The production field mixes producers, distributors, broadcasters, and release labels; boundary-safe string matching is used but this is still a proxy.'
    },
    {
        'definition_family': 'Festival circuit',
        'columns': 'keywords, production',
        'signal': 'Mentions of Cannes, Sundance, Venice, Berlinale, TIFF, Locarno, Rotterdam, Tribeca, Palme d\'Or, Golden Bear, Golden Lion, or generic festival terms.',
        'caveat': 'Festival absence does not mean no festival history; many festival premieres are not encoded.'
    },
    {
        'definition_family': 'Auteur / formal mode',
        'columns': 'keywords, directors, writers',
        'signal': 'Keywords such as art film, avant-garde, experimental film, slow cinema, surrealism, existentialism, social realism; director/writer overlap as a weak auteur proxy.',
        'caveat': 'Narrative ambiguity and style are not directly observable in the dataset.'
    },
    {
        'definition_family': 'Low-budget non-commercial',
        'columns': 'budget, tmdb_popularity, numberOfVotes',
        'signal': 'Positive budget up to 5 million USD, low TMDb popularity, or low IMDb votes.',
        'caveat': 'Budget has many zeros/missing values; zero is treated as unknown, not as a real zero budget.'
    },
    {
        'definition_family': 'Foreign-language / non-Hollywood',
        'columns': 'firstLanguage, original_language, mainCountry, production_countries',
        'signal': 'First/original language is not English and main country is not US.',
        'caveat': 'This over-includes ordinary domestic comedies, TV films, and local genre films; quality/reach gates are needed.'
    },
    {
        'definition_family': 'Niche prestige',
        'columns': 'imdbRating, numberOfVotes',
        'signal': 'High rating with low vote count, reproducing the provocation PDF logic.',
        'caveat': 'IMDb ratings are user-contributed and skew toward specific audience demographics.'
    },
])
display(signal_map)


In [ ]:
print('Auditable specialty labels used in this notebook:')
display(pd.Series(NOTEBOOK_SPECIALTY_LABELS, name='specialty_label').to_frame())

print('Festival terms used in this notebook:')
display(pd.Series(NOTEBOOK_FESTIVAL_TERMS, name='festival_term').to_frame())

print('Arthouse / provocation keyword terms used in this notebook:')
display(pd.Series(NOTEBOOK_ARTHOUSE_KEYWORDS, name='arthouse_keyword').to_frame())

print('Mainstream-risk keywords subtracted in the composite score:')
display(pd.Series(NOTEBOOK_MAINSTREAM_RISK_KEYWORDS, name='mainstream_risk_keyword').to_frame())


In [ ]:
signals = arthouse_signals(df)
scores = score_arthouse(df)

signal_counts = pd.DataFrame({
    'films': signals.sum().astype(int),
    'share_pct': (signals.mean() * 100).round(2),
}).sort_values('films', ascending=False)

display(signal_counts)


<a id="5-definitions"></a>
## 5. Candidate Definitions

We compare five candidate definitions. Each one is deliberately simple enough to audit.

- **PDF baseline:** exactly reproduces the provocation deck: `imdbRating >= 7.0`, `numberOfVotes <= 33`, and keywords present.
- **Strict canon:** specialty label plus festival signal. This is precise but small and can include popular canon titles.
- **Language/country-based:** non-English, non-US, high-rated, and niche-to-mid reach.
- **Keyword-based:** arthouse/provocation keyword, high-rated, and niche-to-mid reach.
- **Composite score:** weighted evidence score with a tunable threshold.

The final recommendation is an ensemble rather than a single pure theory because no single column captures arthouse cinema well.


In [ ]:
evidence_signal = (
    signals['specialty_label']
    | signals['festival_signal']
    | signals['keyword_signal']
    | signals['low_budget']
)

candidate_masks = {
    'pdf_baseline': (
        df['keywords'].notna()
        & df['imdbRating'].ge(7.0)
        & df['numberOfVotes'].le(pdf_vote_q25)
    ),
    'strict_canon': signals['specialty_label'] & signals['festival_signal'],
    'language_country': (
        signals['non_english_non_us']
        & signals['high_rating']
        & signals['niche_reach_broad']
    ),
    'keyword_based': (
        signals['keyword_signal']
        & signals['high_rating']
        & signals['niche_reach_broad']
    ),
    'composite_score': (
        scores.ge(6)
        & signals['high_rating']
        & signals['niche_reach_broad']
        & evidence_signal
    ),
}

working_mask = is_arthouse(df)
candidate_masks['working_ensemble'] = working_mask

analysis_df = df.copy()
for name, mask in candidate_masks.items():
    analysis_df[name] = mask.fillna(False)
analysis_df['arthouse_score'] = scores
for col in signals.columns:
    analysis_df[col] = signals[col]

candidate_summary = []
for name, mask in candidate_masks.items():
    sub = analysis_df[mask]
    candidate_summary.append({
        'candidate': name,
        'films': int(mask.sum()),
        'share_of_dataset_pct': round(mask.mean() * 100, 2),
        'mean_imdb_rating': round(sub['imdbRating'].mean(), 2),
        'median_votes': round(sub['numberOfVotes'].median(), 0),
        'median_year': round(sub['releaseYear'].median(), 0),
        'specialty_label_pct': round(sub['specialty_label'].mean() * 100, 1),
        'festival_signal_pct': round(sub['festival_signal'].mean() * 100, 1),
        'keyword_signal_pct': round(sub['keyword_signal'].mean() * 100, 1),
        'non_english_non_us_pct': round(sub['non_english_non_us'].mean() * 100, 1),
    })

candidate_summary = pd.DataFrame(candidate_summary).set_index('candidate')
display(candidate_summary)


In [ ]:
# Candidate overlap matrix: row candidate as denominator.
candidate_names = list(candidate_masks.keys())
overlap = pd.DataFrame(index=candidate_names, columns=candidate_names, dtype=float)
for row_name in candidate_names:
    row_mask = candidate_masks[row_name]
    denom = row_mask.sum()
    for col_name in candidate_names:
        both = (row_mask & candidate_masks[col_name]).sum()
        overlap.loc[row_name, col_name] = both / denom * 100 if denom else np.nan

display(overlap.round(1))


<a id="6-comparison"></a>
## 6. Comparative Analysis

The next figures compare subset size, year distribution, country distribution, and rating distribution. They use the same visual style as the existing project notebooks.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_counts = candidate_summary['films'].sort_values()
plot_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of films')
ax.set_title('Candidate Arthouse Definitions: Subset Size')
for i, value in enumerate(plot_counts.values):
    ax.text(value + 25, i, f'{int(value):,}', va='center', fontsize=9)
plt.tight_layout()
save_figure(fig, 'arthouse_definition_candidate_sizes')
plt.show()


In [ ]:
# Year distribution by decade.
year_df = analysis_df.dropna(subset=['releaseYear']).copy()
year_df['decade'] = (year_df['releaseYear'].astype(int) // 10) * 10
recent_year_df = year_df[year_df['decade'].between(1950, 2020)]

decade_counts = pd.DataFrame(index=sorted(recent_year_df['decade'].unique()))
for name, mask in candidate_masks.items():
    decade_counts[name] = recent_year_df.loc[mask.reindex(recent_year_df.index), 'decade'].value_counts().sort_index()
decade_counts = decade_counts.fillna(0).astype(int)

display(decade_counts.tail(10))

fig, ax = plt.subplots(figsize=(12, 6))
for name in candidate_names:
    ax.plot(decade_counts.index, decade_counts[name], marker='o', label=name)
ax.set_xlabel('Release decade')
ax.set_ylabel('Number of films')
ax.set_title('Year Distribution by Candidate Definition')
ax.legend(title='Candidate', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
save_figure(fig, 'arthouse_definition_year_distribution')
plt.show()


In [ ]:
# Country distribution: top countries in the recommended working definition.
top_countries = analysis_df.loc[working_mask, 'mainCountry'].value_counts().head(12).index.tolist()
country_matrix = pd.DataFrame(index=top_countries)
for name, mask in candidate_masks.items():
    country_matrix[name] = analysis_df.loc[mask & analysis_df['mainCountry'].isin(top_countries), 'mainCountry'].value_counts()
country_matrix = country_matrix.fillna(0).astype(int)

country_pct = country_matrix.div(country_matrix.sum(axis=0), axis=1).fillna(0) * 100

display(country_matrix)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(country_pct, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Country Distribution by Candidate Definition\n(% within candidate, top working-definition countries)')
ax.set_xlabel('Candidate')
ax.set_ylabel('Main country')
plt.tight_layout()
save_figure(fig, 'arthouse_definition_country_distribution')
plt.show()


In [ ]:
# Rating distribution. Use long form so each candidate can appear independently.
rating_rows = []
for name, mask in candidate_masks.items():
    for rating in analysis_df.loc[mask, 'imdbRating'].dropna():
        rating_rows.append({'candidate': name, 'imdbRating': rating})
rating_long = pd.DataFrame(rating_rows)

display(rating_long.groupby('candidate')['imdbRating'].describe().round(2))

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=rating_long, x='candidate', y='imdbRating', ax=ax, palette='muted')
ax.set_xlabel('Candidate definition')
ax.set_ylabel('IMDb rating')
ax.set_title('Rating Distribution by Candidate Definition')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
plt.tight_layout()
save_figure(fig, 'arthouse_definition_rating_distribution')
plt.show()


<a id="7-sanity-checks"></a>
## 7. Sanity Checks: Examples, False Positives, False Negatives

There is no hand-labeled arthouse ground truth in the dataset, so the false-positive and false-negative lists below are **heuristic boundary checks**:

- **Top examples** are high-scoring or high-rated titles included by a candidate.
- **Likely false positives** are included titles with warning signs such as high mass reach, mainstream-risk keywords, English/US context without other evidence, or weak evidence.
- **Likely false negatives** are titles excluded by a candidate but included by the recommended ensemble.

These tables are meant for human review, not as definitive labels.


In [ ]:
example_cols = [
    'originalTitle', 'releaseYear', 'mainCountry', 'firstLanguage', 'genres',
    'imdbRating', 'numberOfVotes', 'arthouse_score', 'production', 'keywords'
]

candidate_order = ['pdf_baseline', 'strict_canon', 'language_country', 'keyword_based', 'composite_score']

def compact_examples(frame, n=10):
    cols = [col for col in example_cols if col in frame.columns]
    return frame[cols].head(n).reset_index(drop=True)


def warning_reason(row):
    reasons = []
    if row.get('mainstream_risk', False):
        reasons.append('mainstream-risk keyword')
    if pd.notna(row.get('numberOfVotes')) and row.get('numberOfVotes') > 10000:
        reasons.append('mass-reach votes')
    if not row.get('non_english_non_us', False):
        reasons.append('English/US or missing language-country niche')
    if pd.notna(row.get('imdbRating')) and row.get('imdbRating') < 7.0:
        reasons.append('rating below 7.0')
    if not (row.get('specialty_label', False) or row.get('festival_signal', False) or row.get('keyword_signal', False) or row.get('low_budget', False)):
        reasons.append('thin arthouse evidence')
    return '; '.join(reasons) if reasons else 'boundary case for human review'


def likely_false_positives(name, n=10):
    pool = analysis_df[candidate_masks[name]].copy()
    pool['warning_reason'] = pool.apply(warning_reason, axis=1)
    pool['warning_score'] = (
        pool['mainstream_risk'].astype(int) * 4
        + pool['numberOfVotes'].fillna(0).gt(10000).astype(int) * 3
        + (~pool['non_english_non_us']).astype(int)
        + (~(pool['specialty_label'] | pool['festival_signal'] | pool['keyword_signal'] | pool['low_budget'])).astype(int) * 2
        + pool['imdbRating'].fillna(0).lt(7.0).astype(int) * 3
    )
    pool = pool.sort_values(['warning_score', 'numberOfVotes', 'arthouse_score'], ascending=[False, False, True])
    cols = ['warning_reason'] + [col for col in example_cols if col in pool.columns]
    return pool[cols].head(n).reset_index(drop=True)


def likely_false_negatives(name, n=10):
    pool = analysis_df[(~candidate_masks[name]) & working_mask].copy()
    pool = pool.sort_values(['arthouse_score', 'imdbRating', 'numberOfVotes'], ascending=[False, False, False])
    cols = [col for col in example_cols if col in pool.columns]
    return pool[cols].head(n).reset_index(drop=True)

for name in candidate_order:
    display(Markdown(f'### {name}: top 10 included examples'))
    top = analysis_df[candidate_masks[name]].sort_values(
        ['arthouse_score', 'imdbRating', 'numberOfVotes'],
        ascending=[False, False, False]
    )
    display(compact_examples(top, 10))

    display(Markdown(f'### {name}: 10 likely false positives / boundary cases'))
    display(likely_false_positives(name, 10))

    display(Markdown(f'### {name}: 10 likely false negatives relative to the recommended ensemble'))
    display(likely_false_negatives(name, 10))


<a id="8-recommendation"></a>
## 8. Recommendation and Reusable Function

### Recommended project definition

Use the `working_ensemble` implemented in `src/arthouse.py` as the project's default arthouse flag.

Why this ensemble fits the provocation framing:

1. It preserves the deck's core insight: arthouse is a high-rating / low-reach zone, not simply foreign film or independent film.
2. It narrows the old baseline by requiring either arthouse evidence or non-English/non-US context, reducing pure "tiny-vote high-rating" noise.
3. It adds a strict specialty/festival path for films that are clearly arthouse by industry circulation, even if they are too visible to satisfy the low-vote proxy.
4. It adds a composite path for films with multiple weaker signals: festival/specialty/keyword/budget plus quality and niche-to-mid reach.

The function to reuse elsewhere is:

```python
from src.arthouse import is_arthouse

arthouse_mask = is_arthouse(df)
arthouse_df = df[arthouse_mask]
```


In [ ]:
# Smoke test the reusable function and show the final working subset.
arthouse_mask = is_arthouse(df)
arthouse_df = df[arthouse_mask].copy()

print(f'Working definition selects {arthouse_mask.sum():,} films ({arthouse_mask.mean() * 100:.2f}% of dataset).')
print(f'Mean IMDb rating: {arthouse_df["imdbRating"].mean():.2f}')
print(f'Median IMDb votes: {arthouse_df["numberOfVotes"].median():,.0f}')
print(f'Median release year: {arthouse_df["releaseYear"].median():.0f}')

working_signal_profile = pd.DataFrame({
    'working_ensemble_count': signals[arthouse_mask].sum().astype(int),
    'working_ensemble_pct': (signals[arthouse_mask].mean() * 100).round(1),
    'full_dataset_pct': (signals.mean() * 100).round(1),
})
display(working_signal_profile)

display(
    arthouse_df.assign(arthouse_score=scores[arthouse_mask])
    .sort_values(['arthouse_score', 'imdbRating', 'numberOfVotes'], ascending=[False, False, False])
    [example_cols]
    .head(20)
    .reset_index(drop=True)
)


In [ ]:
# Save a compact CSV of the recommended titles for easy downstream inspection.
# This is an analysis artifact, not the canonical definition; src/arthouse.py is canonical.
out_cols = [
    'titleId', 'originalTitle', 'releaseYear', 'mainCountry', 'firstLanguage',
    'genres', 'imdbRating', 'numberOfVotes', 'arthouse_score'
]
export_df = arthouse_df.assign(arthouse_score=scores[arthouse_mask])
export_path = Path('reports/arthouse_working_definition_titles.csv')
export_df[[col for col in out_cols if col in export_df.columns]].to_csv(export_path, index=False)
print(f'Saved {export_path} with {len(export_df):,} rows for inspection.')


<a id="9-limitations"></a>
## 9. Open Questions and Known Limitations

- **No ground truth labels:** The false-positive and false-negative lists are sanity checks, not validation against human-coded arthouse labels.
- **Production field ambiguity:** `production` mixes production companies, distributors, broadcasters, and release labels. Specialty-label hits are useful but not pure distribution evidence.
- **Festival evidence is sparse:** Festival history is rarely encoded systematically, so the strict canon definition is precise but under-inclusive.
- **IMDb vote count is a proxy:** Low votes can mean niche prestige, but it can also mean poor data coverage, very recent release, or local-only visibility.
- **IMDb ratings are biased:** Ratings reflect IMDb users, not festival juries, critics, Publikum's audience panels, or Danish arthouse cinemagoers.
- **Budget data is incomplete:** TMDb budget zeros are treated as unknown, so low-budget evidence only applies when a positive budget exists.
- **Language/country over-includes:** Non-English/non-US cinema contains mainstream local entertainment as well as arthouse. The recommended definition therefore combines language/country with rating, reach, and evidence signals.
- **Formal style is mostly invisible:** Key film-studies ideas such as ambiguity, realism, slow pacing, and authorial expressivity are only approximated through keywords and writer/director overlap.

### Practical next steps

Use `is_arthouse(df)` as the default segmentation in downstream notebooks. If a later qualitative review produces hand-labeled examples, recalibrate the score threshold and the keyword/specialty lists against that review.
